# Modules

In [2]:
import dspy
import os

GEMINI_API_KEY = os.environ.get(key='GEMINI_API_KEY')

dspy.configure(lm=dspy.LM(
    model="gemini/gemini-2.5-flash",
    temperature=0.6
))

sentence = "it's a charming and often affecting journey."  # example from the SST-2 dataset.

# 1) Declare with a signature.
classify = dspy.Predict('sentence -> sentiment: bool')

# 2) Call with input argument(s). 
response = classify(sentence=sentence)

# 3) Access the output.
print(response.sentiment)

True


In [3]:
question = "What's something great about the ColBERT retrieval model?"

# 1) Declare with a signature, and pass some config.
classify = dspy.ChainOfThought('question -> answer', n=5)

# 2) Call with input argument.
response = classify(question=question)
print(response)
# 3) Access the outputs.
print(response.completions.answer)

print(f"Reasoning: {response.reasoning}")
print(f"Answer: {response.answer}")

response.completions[3].reasoning == response.completions.reasoning[3]

Prediction(
    reasoning='The user is asking for a significant advantage of the ColBERT retrieval model. ColBERT\'s core innovation lies in its "late interaction" mechanism. Unlike traditional dense retrieval models that compress a document into a single vector, ColBERT generates multiple contextualized embeddings for each document token. This allows for a more fine-grained, token-level matching between query and document at retrieval time. This approach maintains much of the semantic richness of BERT-based models while being significantly more efficient than full cross-encoders because document embeddings can be pre-computed and indexed.\n\nTherefore, a great aspect is its ability to perform highly accurate, fine-grained matching between queries and documents at the token level, leading to high retrieval quality, while still being very efficient and scalable due to pre-computation and late interaction.',
    answer='ColBERT excels at performing highly accurate, fine-grained, token-le

True

### What other DSPy modules are there? How can I use them?
The others are very similar. They mainly change the internal behavior with which your signature is implemented!

1. `dspy.Predict`: Basic predictor. Does not modify the signature. Handles the key forms of learning (i.e., storing the instructions and demonstrations and updates to the LM).

2. `dspy.ChainOfThought`: Teaches the LM to think step-by-step before committing to the signature's response.

3. `dspy.ProgramOfThought`: Teaches the LM to output code, whose execution results will dictate the response.

4. `dspy.ReAct`: An agent that can use tools to implement the given signature.

5. `dspy.MultiChainComparison`: Can compare multiple outputs from ChainOfThought to produce a final prediction.

In [4]:
math = dspy.ChainOfThought("question -> answer: float")
math(question="Two dice are tossed. What is the probability that the sum equals two?")

Prediction(
    reasoning='To find the probability, we need to determine the number of favorable outcomes and the total number of possible outcomes.\n\n1.  **Total Possible Outcomes:** When two dice are tossed, each die has 6 faces (1, 2, 3, 4, 5, 6). The total number of possible combinations is 6 * 6 = 36. These combinations can be represented as ordered pairs (die1_result, die2_result).\n\n2.  **Favorable Outcomes:** We are looking for the sum of the two dice to be equal to two.\n    *   The minimum value a single die can show is 1.\n    *   Therefore, the only way to get a sum of 2 is if both dice show a 1.\n    *   This outcome is (1, 1).\n    *   There is only 1 favorable outcome.\n\n3.  **Calculate Probability:** Probability is calculated as (Number of Favorable Outcomes) / (Total Number of Possible Outcomes).\n    *   Probability = 1 / 36\n\n4.  **Convert to float:** 1/36 as a float is approximately 0.027777777777777776.',
    answer=0.027777777777777776
)

In [9]:
def search(query: str, k: int = 3) -> list[str]:
    """Retrieves abstracts from Wikipedia."""
    results = dspy.ColBERTv2(url='http://20.102.90.50:2017/wiki17_abstracts')(query, k=k)
    return [x['text'] for x in results]


In [ ]:
from typing import Literal

class Classify(dspy.Signature):
    """Classify sentiment of a given sentence."""

    sentence: str = dspy.InputField()
    sentiment: Literal['positive', 'negative', 'neutral'] = dspy.OutputField()
    confidence: float = dspy.OutputField()

classify = dspy.Predict(Classify)
classify(sentence="This book was super fun to read, though not the last chapter.")

Prediction(
    sentiment='neutral',
    confidence=0.85
)

In [ ]:
text = "Apple Inc. announced its latest iPhone 14 today. The CEO, Tim Cook, highlighted its new features in a press release."

module = dspy.Predict("text -> title, headings: list[str], entities_and_metadata: list[dict[str, str]]")
response = module(text=text)

print(response.title)
print(response.headings)
print(response.entities_and_metadata)

Apple Inc. Announces iPhone 14
[]
[{'entity_name': 'Apple Inc.', 'entity_type': 'ORGANIZATION', 'role': 'Manufacturer'}, {'entity_name': 'iPhone 14', 'entity_type': 'PRODUCT', 'manufacturer': 'Apple Inc.'}, {'entity_name': 'Tim Cook', 'entity_type': 'PERSON', 'title': 'CEO', 'organization': 'Apple Inc.'}]


In [ ]:
def evaluate_math(expression: str) -> float:
    return dspy.PythonInterpreter({}).execute(expression)

def search_wikipedia(query: str) -> str:
    results = dspy.ColBERTv2(url='http://20.102.90.50:2017/wiki17_abstracts')(query, k=3)
    return [x['text'] for x in results]

react = dspy.ReAct("question -> answer: float", tools=[evaluate_math, search_wikipedia])

pred = react(question="What is 9362158 divided by the year of birth of David Gregory of Kinnairdy castle?")
print(pred.answer)

nan


In [12]:
class Hop(dspy.Module):
    def __init__(self, num_docs=10, num_hops=4):
        self.num_docs, self.num_hops = num_docs, num_hops
        self.generate_query = dspy.ChainOfThought('claim, notes -> query')
        self.append_notes = dspy.ChainOfThought('claim, notes, context -> new_notes: list[str], titles: list[str]')

    def forward(self, claim: str) -> list[str]:
        notes = []
        titles = []

        for _ in range(self.num_hops):
            query = self.generate_query(claim=claim, notes=notes).query
            context = search(query,k=self.num_docs)
            prediction = self.append_notes(claim=claim, notes=notes, context=context)
            notes.extend(prediction.new_notes)
            titles.extend(prediction.titles)

        return dspy.Prediction(notes=notes, titles=list(set(titles)))

hop = Hop()
print(hop(claim="Stephen Curry is the best 3 pointer shooter ever in the human history"))

KeyError: 'topk'

In [ ]:
dspy.settings.configure(track_usage=True)

In [ ]:
dspy.settings.configure(track_usage=True)

In [20]:
usage = prediction_instance.get_lm_usage()

NameError: name 'prediction_instance' is not defined

In [22]:
import dspy

# Configure DSPy with tracking enabled
dspy.settings.configure(
    lm=dspy.LM("gemini/gemini-2.5-flash", cache=False),
    track_usage=True
)

# Define a simple program that makes multiple LM calls
class MyProgram(dspy.Module):
    def __init__(self):
        self.predict1 = dspy.ChainOfThought("question -> answer")
        self.predict2 = dspy.ChainOfThought("question, answer -> score")

    def __call__(self, question: str) -> str:
        answer = self.predict1(question=question)
        score = self.predict2(question=question, answer=answer)
        return score

# Run the program and check usage
program = MyProgram()
output = program(question="What is the capital of France?")
print(output.get_lm_usage())

{'gemini/gemini-2.5-flash': {'completion_tokens': 97, 'prompt_tokens': 247, 'total_tokens': 344, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 53, 'rejected_prediction_tokens': None, 'text_tokens': 44}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 247, 'image_tokens': None}}}


In [23]:
# Enable caching
dspy.settings.configure(
    lm=dspy.LM("gemini/gemini-2.5-flash", cache=True),
    track_usage=True
)

program = MyProgram()

# First call - will show usage statistics
output = program(question="What is the capital of Zambia?")
print(output.get_lm_usage())  # Shows token usage

# Second call - same question, will use cache
output = program(question="What is the capital of Zambia?")
print(output.get_lm_usage())  # Shows empty dict: {}

{'gemini/gemini-2.5-flash': {'completion_tokens': 97, 'prompt_tokens': 248, 'total_tokens': 345, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 55, 'rejected_prediction_tokens': None, 'text_tokens': 42}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None, 'text_tokens': 248, 'image_tokens': None}}}
{}
